# Interview Prep Simulation Scenarios: Machine Learning Paradigms

Your Name

---

## Getting Started

- Replace **Your Name** above with your full name
- Automatic 0 if you include your student ID or any other identifying number
- Rename the file to `Your_Name.ipynb` before submitting
- When finished, share your Colab link with **Anyone with the link can view** privileges
- Paste the shared link into the Canvas submission

**Grading:** Pass / Fail — complete all sections with written justification

---

## How to Use This Notebook

This notebook is designed to be accessible to all users, including those navigating with a **screen reader**.

### Screen Reader Navigation Guide

Every code section in this notebook gives you a choice. Before each code cell you will find a navigation landmark with two options:

- **Skip this code cell** — Jumps past the code directly to the output explanation.
- **Go back and read the code cell** — Returns you to the top of the code section.

**Tips for screen reader users:**
- Press **H** to jump between major section headings
- Press **K** or **Tab** to move between links
- Press **D** to jump between landmark regions

---

## Learning Objectives

By the end of this assignment you will be able to:

1. Diagnose violations of linear regression assumptions from residual plots
2. Explain why accuracy is a misleading metric for imbalanced classification
3. Calculate and compare business cost under different decision thresholds
4. Interpret Random Forest feature importances and explain their directional limitations
5. Scale features and apply PCA before clustering
6. Use the elbow method and silhouette score to select K for K-Means
7. Assign business-meaningful names to discovered clusters
8. Distinguish K-Means from DBSCAN and explain when each is appropriate
9. Implement the Q-Learning Bellman update
10. Explain Transfer Learning and Federated Learning to a non-technical audience

---

## Notebook Overview

| Part | Company | Paradigm | Tasks |
| :--- | :--- | :--- | :--- |
| **1** | SportsPulse Analytics | Supervised Learning | Regression, Classification, Thresholding, Random Forest |
| **2** | SportsPulse Fan Intelligence | Unsupervised Learning | PCA, K-Means, Cluster Naming, DBSCAN |
| **3** | SolarVault | Reinforcement Learning | Q-Learning, Transfer Learning, Federated Learning |

---

# Part 1: Supervised Learning

## Scenario

You have passed the initial phone screen for a Data Scientist role at **SportsPulse Analytics**, a sports science firm that works with professional teams to reduce athlete injury rates and optimize performance.

**Message from the Hiring Manager:**

> 'We have attached synthetic data representing two core problems: estimating an athlete's return-to-play timeline after injury (Regression) and predicting whether an athlete is at high risk for a season-ending event (Classification).

> We do not care about hitting 99% accuracy. We care about **how you think**.
> 1. Check your assumptions.
> 2. Tell us what the model outputs *mean* in plain language.
> 3. Make a recommendation the medical team can act on.'

## Instructions

1. Run the Data Generation cells — do not modify them
2. Complete every **TODO** section
3. Answer every **Interview Question** in the Markdown cell provided

---

<a name="read-code-1"></a>

**Setup Cell — Import Libraries (Part 1)**

This cell imports all libraries needed for Part 1: `pandas` and `numpy` for data handling, `matplotlib` and `seaborn` for visualization, and scikit-learn modules for regression, classification, ensemble models, and evaluation metrics.

<nav aria-label="Code cell 1 navigation">
<a href="#skip-code-1" aria-label="Skip code cell 1 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (mean_squared_error, classification_report,
                             confusion_matrix, precision_recall_curve)

np.random.seed(101)
sns.set_style('whitegrid')
%matplotlib inline
print('Part 1 setup complete.')

<a name="skip-code-1"></a>

**Expected output:** `Part 1 setup complete.`

If any import fails, install the missing package with `!pip install <package_name>` and re-run.

<nav aria-label="Return navigation for code cell 1">
<a href="#read-code-1" aria-label="Go back and read code cell 1">&#8617; Go back and read the code cell</a>
</nav>

---
## Task 1: Regression — Athlete Return-to-Play Estimation

**Context:** SportsPulse wants to predict how many days an injured athlete will need before returning to play, based on three factors:

1. `Injury_Severity` — Clinician rating from 1 (minor) to 10 (career-threatening)
2. `Weekly_Training_Load` — Average hours of weekly training in the month before injury
3. `Athlete_Age` — Age in years

**The Catch:** Recovery time is not linear with age. Each additional year of age compounds the recovery burden exponentially — a 35-year-old takes dramatically longer to recover than a 25-year-old, not just proportionally longer. A standard Linear Regression will miss this.

<a name="read-code-2"></a>

**Data Generation Cell — Athlete Recovery Dataset (Do Not Modify)**

This cell generates 1,000 synthetic athlete records. The true relationship is: `Recovery_Days = 10 + 8 × Severity + 2 × TrainingLoad + exp(0.4 × Age/10)`. The exponential term on `Athlete_Age` is the non-linearity that will show up as a pattern in your residual plot. A noise term drawn from N(0, 8) simulates real-world measurement error.

<nav aria-label="Code cell 2 navigation">
<a href="#skip-code-2" aria-label="Skip code cell 2 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
n = 1000
severity     = np.random.uniform(1, 10, n)
training_load = np.random.uniform(5, 30, n)
age          = np.random.uniform(18, 40, n)

# True relationship: recovery days has an exponential component on age
recovery_days = (10 + 8 * severity + 2 * training_load
                 + np.exp(0.4 * age / 10) + np.random.normal(0, 8, n))

df_athletes = pd.DataFrame({
    'Injury_Severity': severity,
    'Weekly_Training_Load': training_load,
    'Athlete_Age': age,
    'Recovery_Days': recovery_days
})

X_reg = df_athletes[['Injury_Severity', 'Weekly_Training_Load', 'Athlete_Age']]
y_reg = df_athletes['Recovery_Days']
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=101)

print(f'Dataset shape: {df_athletes.shape}')
print('\nFirst 5 rows:')
print(df_athletes.head())

<a name="skip-code-2"></a>

**Expected output:** Shape (1000, 4) and a 5-row preview. `Recovery_Days` values should range roughly from 30 to 120 days, with older, more severely injured athletes at the high end.

<nav aria-label="Return navigation for code cell 2">
<a href="#read-code-2" aria-label="Go back and read code cell 2">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-3"></a>

**Task 1 Code Cell — Fit Linear Regression and Plot Residuals**

Step 1: Instantiate `LinearRegression`, fit to `X_train_r` and `y_train_r`, and predict on `X_test_r`. Step 2: Compute RMSE (square root of `mean_squared_error`). Step 3: Plot Predicted Values on the x-axis against Residuals (Actual minus Predicted) on the y-axis. Add a horizontal dashed line at y = 0. If the residuals form a U-shape or fan shape instead of a random cloud, the linear model is missing something.

<nav aria-label="Code cell 3 navigation">
<a href="#skip-code-3" aria-label="Skip code cell 3 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# TODO: Fit a Standard Linear Regression Model
# 1. Instantiate and fit LinearRegression
# [YOUR CODE HERE]

# 2. Predict on X_test_r
# [YOUR CODE HERE]

# 3. Calculate and print RMSE
# rmse = np.sqrt(mean_squared_error(y_test_r, preds_r))
# print(f'Linear Regression RMSE: {rmse:.2f} days')

# TODO: Residual Plot
# residuals = y_test_r - preds_r
# plt.figure(figsize=(10, 5))
# plt.scatter(preds_r, residuals, alpha=0.4, color='steelblue', edgecolors='none')
# plt.axhline(0, color='red', linestyle='--', linewidth=2)
# plt.xlabel('Predicted Recovery Days')
# plt.ylabel('Residuals (Actual - Predicted)')
# plt.title('Residual Plot — Linear Regression on Athlete Recovery')
# plt.grid(alpha=0.3)
# plt.tight_layout()
# plt.savefig('fig_residuals.png', dpi=150, bbox_inches='tight')
# plt.show()

<a name="skip-code-3"></a>

**Expected output:** An RMSE value (higher than you would want in production — the linear model is imperfect by design) and a residual plot. The plot should show a curved or fan-shaped pattern rather than a random cloud, indicating that the linear model systematically under- or over-predicts for certain ranges of the predictor variables.

**Accessibility note:** Add a Markdown cell below describing the residual plot. Note whether the pattern is a U-shape, funnel, or random cloud, and what that implies about the model's fit.

<nav aria-label="Return navigation for code cell 3">
<a href="#read-code-3" aria-label="Go back and read code cell 3">&#8617; Go back and read the code cell</a>
</nav>

## Interview Question 1

**The Hiring Manager asks:** *'Your residual plot shows a clear curved pattern rather than a random cloud. What does this tell us about the assumptions of OLS Linear Regression? How would you fix this model to better capture the age-recovery dynamic?'*

**Your Answer:**

- **Observation:** *[Describe the pattern you see in the residual plot]*

- **Violation:** *[Which OLS assumption is violated? Linearity? Homoscedasticity? Explain the difference.]*

- **Fix:** *[Choose one: log-transform Recovery_Days, add Athlete_Age² as a polynomial feature, or use a Generalized Linear Model (Gamma GLM). Justify your choice.]*

---
**YOUR ANSWER HERE** — replace the italicized lines.

---
## Task 2: Classification — High-Risk Injury Prediction

**Context:** The medical team wants to flag athletes who are at high risk for a season-ending injury before it happens. These events are rare — about 10% of athletes in any given season experience one.

**The Catch:** The data is severely **imbalanced**. 90% of athletes are healthy (Class 0); only 10% will have a season-ending event (Class 1). A naive model that predicts 'healthy' for everyone will score 90% accuracy while being completely useless.

<a name="read-code-4"></a>

**Data Generation Cell — High-Risk Injury Dataset (Do Not Modify)**

This cell uses `make_classification` to generate 1,000 athlete records with 5 features: `Avg_Training_Intensity`, `Recovery_Score`, `Sleep_Hours`, `Injury_History_Count`, and a fifth synthetic feature. The class weights are set to 90% healthy / 10% high-risk. The data is split with stratification to preserve the class imbalance in both sets.

<nav aria-label="Code cell 4 navigation">
<a href="#skip-code-4" aria-label="Skip code cell 4 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.datasets import make_classification

X_cls, y_cls = make_classification(
    n_samples=1000, n_features=5, n_informative=3, n_redundant=1,
    n_classes=2, weights=[0.9, 0.1], flip_y=0.01, random_state=42
)

cols = ['Avg_Training_Intensity', 'Recovery_Score', 'Sleep_Hours',
        'Injury_History_Count', 'Feature_5']
df_injury = pd.DataFrame(X_cls, columns=cols)
df_injury['High_Risk'] = y_cls

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    df_injury[cols], df_injury['High_Risk'],
    test_size=0.3, stratify=df_injury['High_Risk'], random_state=42
)

print(f'Dataset shape: {df_injury.shape}')
print('\nClass distribution:')
print(df_injury['High_Risk'].value_counts(normalize=True).round(3))

<a name="skip-code-4"></a>

**Expected output:** Shape (1000, 6). Class distribution should show approximately 0.9 for Class 0 and 0.1 for Class 1, confirming the 90/10 imbalance.

<nav aria-label="Return navigation for code cell 4">
<a href="#read-code-4" aria-label="Go back and read code cell 4">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-5"></a>

**Task 2 Code Cell — Logistic Regression and Confusion Matrix**

Step 1: Instantiate `LogisticRegression()` and fit to the training data. Step 2: Print the accuracy score on the test set. Step 3: Print the full confusion matrix using `confusion_matrix(y_test_c, predictions)`.

<nav aria-label="Code cell 5 navigation">
<a href="#skip-code-5" aria-label="Skip code cell 5 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# TODO: Fit a Logistic Regression Model
# 1. Fit LogisticRegression to X_train_c, y_train_c
# [YOUR CODE HERE]

# 2. Print accuracy on the test set
# print(f'Accuracy: {model_log.score(X_test_c, y_test_c):.3f}')

# TODO: Confusion Matrix
# predictions = model_log.predict(X_test_c)
# cm = confusion_matrix(y_test_c, predictions)
# print('\nConfusion Matrix:')
# print(cm)
# print('\nClassification Report:')
# print(classification_report(y_test_c, predictions,
#       target_names=['Healthy (0)', 'High-Risk (1)']))

<a name="skip-code-5"></a>

**Expected output:** Accuracy around 0.90 — which sounds impressive. But the confusion matrix will show very few actual high-risk athletes caught (low recall for Class 1). This is the core lesson: high accuracy in imbalanced datasets is a warning sign, not a victory.

<nav aria-label="Return navigation for code cell 5">
<a href="#read-code-5" aria-label="Go back and read code cell 5">&#8617; Go back and read the code cell</a>
</nav>

## Interview Question 2

**The Hiring Manager asks:** *'You reported 90%+ accuracy. That sounds incredible! But look at the confusion matrix — how many actual high-risk athletes did we catch? Why is accuracy a dangerous metric for this specific problem?'*

**Your Answer:**

- *[How many True Positives (actual high-risk athletes correctly flagged) does the matrix show?]*

- *[Explain the concept of a Null Classifier — a model that predicts 'Healthy' for everyone would also achieve ~90% accuracy. What does this tell you about accuracy as a metric here?]*

- *[What metric should you use instead? Recall? F1? Precision-Recall AUC? Why is Recall for Class 1 the metric the medical team actually cares about?]*

---
**YOUR ANSWER HERE**

---
## Task 3: The Business Trade-off — Thresholding

**Context:** Not all errors cost the same.

- A **False Positive** (flagging a healthy athlete as high-risk) triggers an unnecessary medical evaluation: **\$200 cost**.
- A **False Negative** (missing a true high-risk athlete) means the injury goes unaddressed, leading to a season-ending event, rehabilitation costs, and potential contract breach: **\$50,000 cost**.

These costs are wildly asymmetric. We need to lower the decision threshold to catch more true high-risk athletes, even if we flag more false alarms.

<a name="read-code-6"></a>

**Task 3 Code Cell — Compare Business Cost at Two Decision Thresholds**

This cell fits a Logistic Regression and retrieves predicted probabilities for the positive class. It then computes the total business cost under two thresholds: the default of 0.5 and a more aggressive threshold of 0.2. Cost is calculated as: `(False Positives × $200) + (False Negatives × $50,000)`. The cell is mostly complete — it runs as-is once your model from Task 2 exists. Read the output carefully before answering Interview Question 3.

<nav aria-label="Code cell 6 navigation">
<a href="#skip-code-6" aria-label="Skip code cell 6 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
model_log = LogisticRegression(random_state=42)
model_log.fit(X_train_c, y_train_c)
probs = model_log.predict_proba(X_test_c)[:, 1]

# Cost parameters
FP_COST = 200      # Unnecessary medical evaluation
FN_COST = 50000    # Missed high-risk athlete: injury + rehab + contract

# Default Threshold (0.5)
preds_default = (probs > 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test_c, preds_default).ravel()
cost_default = (fp * FP_COST) + (fn * FN_COST)

# Aggressive Threshold (0.2) — catch more high-risk athletes
preds_aggressive = (probs > 0.2).astype(int)
tn_a, fp_a, fn_a, tp_a = confusion_matrix(y_test_c, preds_aggressive).ravel()
cost_aggressive = (fp_a * FP_COST) + (fn_a * FN_COST)

print('=== Threshold Comparison ===')
print(f'\nThreshold 0.5 (default):')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print(f'  Business Cost: ${cost_default:,}')

print(f'\nThreshold 0.2 (aggressive):')
print(f'  TP={tp_a}  FP={fp_a}  FN={fn_a}  TN={tn_a}')
print(f'  Business Cost: ${cost_aggressive:,}')

print(f'\nCost Reduction from Lowering Threshold: ${cost_default - cost_aggressive:,}')

<a name="skip-code-6"></a>

**Expected output:** Two cost comparisons. The aggressive threshold (0.2) will have more False Positives (more unnecessary evaluations) but dramatically fewer False Negatives (fewer missed high-risk athletes). Given the \$50,000 cost of a missed case vs \$200 for a false alarm, the lower threshold should produce a significantly lower total business cost.

<nav aria-label="Return navigation for code cell 6">
<a href="#read-code-6" aria-label="Go back and read code cell 6">&#8617; Go back and read the code cell</a>
</nav>

## Interview Question 3

**The Hiring Manager asks:** *'Based on the costs above, which deployment strategy do you recommend — the conservative threshold (0.5) or the aggressive one (0.2)? Walk me through your reasoning.'*

**Your Answer:**

- **Recommendation:** *[State your threshold choice]*

- **ROI Justification:** *[Calculate or describe the cost saving. How many missed high-risk athletes does the aggressive threshold prevent, and what is the cost per prevented case?]*

- **Limitations:** *[Are there any situations where the conservative threshold would be preferable? Think about budget constraints or high false-alarm fatigue among medical staff.]*

---
**YOUR ANSWER HERE**

---
## Task 4: Random Forest and Feature Importance

**Context:** The sports science team wants to know *which* factors drive injury risk so they can build prevention programs. Logistic Regression coefficients require scaled features to be interpretable. Random Forest feature importances work regardless of scale, but they have an important limitation.

<a name="read-code-7"></a>

**Task 4 Code Cell — Random Forest Feature Importance**

Step 1: Fit a `RandomForestClassifier` with `n_estimators=100` and `random_state=42` to the injury classification training data. Step 2: Extract `.feature_importances_` and pair them with the column names. Step 3: Create a horizontal bar chart sorted from most to least important. Add a comment explaining what feature importance measures and why it cannot tell you the *direction* of the relationship.

<nav aria-label="Code cell 7 navigation">
<a href="#skip-code-7" aria-label="Skip code cell 7 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# TODO: Fit a Random Forest Classifier
# rf = RandomForestClassifier(n_estimators=100, random_state=42)
# rf.fit(X_train_c, y_train_c)
# [YOUR CODE HERE]

# TODO: Extract and plot feature importances
# importances = pd.Series(rf.feature_importances_, index=cols).sort_values()
# plt.figure(figsize=(10, 5))
# importances.plot(kind='barh', color='steelblue', alpha=0.8, edgecolor='black')
# plt.xlabel('Feature Importance (Mean Decrease in Impurity)')
# plt.title('Random Forest — Injury Risk Feature Importance')
# plt.grid(axis='x', alpha=0.3)
# plt.tight_layout()
# plt.savefig('fig_feature_importance.png', dpi=150, bbox_inches='tight')
# plt.show()

# TODO: Print RF accuracy vs Logistic Regression accuracy
# print(f'Random Forest Accuracy:   {rf.score(X_test_c, y_test_c):.3f}')
# print(f'Logistic Regression Acc:  {model_log.score(X_test_c, y_test_c):.3f}')

<a name="skip-code-7"></a>

**Expected output:** A horizontal bar chart and two accuracy scores. Random Forest should score similarly to or slightly better than Logistic Regression. The chart shows which features the forest found most informative for splitting decisions.

**Accessibility note:** Add a Markdown cell below describing the bar chart. Note which feature ranks highest and what that implies for sports medicine program design.

<nav aria-label="Return navigation for code cell 7">
<a href="#read-code-7" aria-label="Go back and read code cell 7">&#8617; Go back and read the code cell</a>
</nav>

## Interview Question 4

**The Hiring Manager asks:** *'Your Random Forest performs slightly better, but it is a black box. If the Head of Sports Medicine asks whether more Injury_History_Count increases or decreases high-risk probability — does feature importance answer that question? How would you derive the directionality?'*

**Your Answer:**

- **What feature importance does measure:** *[Describe mean decrease in impurity — how often the feature is used for splitting and how much it reduces node impurity on average]*

- **What it does NOT measure:** *[Feature importance cannot tell you whether the relationship is positive or negative]*

- **How to get directionality:** *[Choose one of: check a Partial Dependence Plot, compute SHAP values, or simply look at the correlation matrix as a quick sanity check. Explain your choice.]*

---
**YOUR ANSWER HERE**

---
# Part 2: Unsupervised Learning

## Scenario

**SportsPulse Fan Intelligence** — the marketing division — has a problem: they treat every fan the same. Premium subscribers, casual viewers, merchandise collectors, and social media amplifiers all receive the same generic promotional email.

**Message from the Head of Fan Marketing:**

> 'We have behavioral data on 700 fans from the past season. We do not know who these people are as types — no labels exist. But we suspect there are at least 3 distinct fan segments.

> 1. Find the segments.
> 2. Give each one a name that means something to the marketing team.
> 3. We also suspect that some accounts in our data are bots — they inflate our engagement metrics. Can you identify them?'

---

<a name="read-code-8"></a>

**Setup Cell — Import Libraries (Part 2)**

This cell imports everything needed for Part 2: `StandardScaler` for feature scaling, `PCA` for dimensionality reduction, `KMeans` and `DBSCAN` for clustering, and `silhouette_score` for cluster quality evaluation.

<nav aria-label="Code cell 8 navigation">
<a href="#skip-code-8" aria-label="Skip code cell 8 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

np.random.seed(42)
sns.set_style('whitegrid')
%matplotlib inline
print('Part 2 setup complete.')

<a name="skip-code-8"></a>

**Expected output:** `Part 2 setup complete.`

<nav aria-label="Return navigation for code cell 8">
<a href="#read-code-8" aria-label="Go back and read code cell 8">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-9"></a>

**Data Generation Cell — Fan Behavior Dataset (Do Not Modify)**

This cell generates 700 fan records across four distinct behavioral groups:

- **Casual Viewers (300):** Moderate session time, few purchases, low engagement depth
- **Super Fans (200):** Long session time, high merchandise spend, high event attendance
- **Disengaged / At-Risk (150):** Short sessions, high support contacts, low spend
- **Bots (50):** Near-zero session time, extremely high content requests, zero spend

The five features are: `Avg_Session_Hrs`, `Content_Requests`, `Merch_Spend_USD`, `Events_Attended`, `Support_Contacts`.

<nav aria-label="Code cell 9 navigation">
<a href="#skip-code-9" aria-label="Skip code cell 9 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Fan behavioral segments
data_casual    = np.random.normal(loc=[1.5, 20,  50,  2,  1], scale=[0.5, 5,  20,  1, 0.5], size=(300, 5))
data_superfans = np.random.normal(loc=[4.0, 10, 500, 12,  0], scale=[1.0, 2,  50,  2, 0.2], size=(200, 5))
data_disengaged= np.random.normal(loc=[0.5,  5,  15,  0,  8], scale=[0.3, 2,  10,  0, 2.0], size=(150, 5))
data_bots      = np.random.normal(loc=[0.02,150,   0,  0,  0], scale=[0.01,15,   0,  0, 0.0], size=(50,  5))

data = np.vstack([data_casual, data_superfans, data_disengaged, data_bots])
df_fans = pd.DataFrame(data, columns=[
    'Avg_Session_Hrs', 'Content_Requests', 'Merch_Spend_USD', 'Events_Attended', 'Support_Contacts'
])
df_fans = df_fans.sample(frac=1).reset_index(drop=True)

print(f'Dataset shape: {df_fans.shape}')
print('\nFirst 5 rows:')
print(df_fans.head())

<a name="skip-code-9"></a>

**Expected output:** Shape (700, 5) and a 5-row preview. Notice the very different scales: `Merch_Spend_USD` ranges into the hundreds while `Avg_Session_Hrs` is near 0–5. This scale difference is exactly why we must standardize before K-Means.

<nav aria-label="Return navigation for code cell 9">
<a href="#read-code-9" aria-label="Go back and read code cell 9">&#8617; Go back and read the code cell</a>
</nav>

---
## Task 1: Preprocessing and PCA

**Context:** The five features are on completely different scales. `Merch_Spend_USD` is in the hundreds while `Avg_Session_Hrs` is 0 to 5. K-Means uses Euclidean distance — without scaling, `Merch_Spend_USD` will dominate every distance calculation and the other features will be effectively ignored.

**Interview Logic:** You cannot visualize 5 dimensions simultaneously. After scaling, reduce to 2 dimensions with PCA so you can *see* the cluster structure before committing to a value of K.

<a name="read-code-10"></a>

**Task 1 Code Cell — Scale Features and Apply PCA**

Step 1: Initialize a `StandardScaler`, fit to `df_fans`, and save the result as `X_scaled`. Step 2: Initialize `PCA(n_components=2)`, fit and transform `X_scaled`, save as `X_pca`. Step 3: Print the explained variance ratio for both components. Step 4: Create a scatter plot of PC1 (x-axis) vs PC2 (y-axis) to visualize the structure.

<nav aria-label="Code cell 10 navigation">
<a href="#skip-code-10" aria-label="Skip code cell 10 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# TODO: Scale and PCA
# Step 1: StandardScaler
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(df_fans)
# [YOUR CODE HERE]

# Step 2: PCA to 2D
# pca = PCA(n_components=2)
# X_pca = pca.fit_transform(X_scaled)
# [YOUR CODE HERE]

# Step 3: Print variance explained
# print('Explained variance ratio:', pca.explained_variance_ratio_)
# print(f'Total variance retained: {sum(pca.explained_variance_ratio_)*100:.1f}%')

# Step 4: Scatter plot
# plt.figure(figsize=(10, 7))
# plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.4, s=20, edgecolors='none')
# plt.xlabel('Principal Component 1')
# plt.ylabel('Principal Component 2')
# plt.title('Fan Behavior — PCA 2D Projection')
# plt.grid(alpha=0.3)
# plt.tight_layout()
# plt.savefig('fig_pca_fans.png', dpi=150, bbox_inches='tight')
# plt.show()

<a name="skip-code-10"></a>

**Expected output:** Two variance ratio values summing to between 0.70 and 0.90, and a scatter plot showing visible cluster structure. The bot cluster should appear as a tight, isolated group far from the main data mass.

**Accessibility note:** Add a Markdown cell below describing what the scatter plot shows — how many apparent clusters are visible and whether any isolated groups appear.

<nav aria-label="Return navigation for code cell 10">
<a href="#read-code-10" aria-label="Go back and read code cell 10">&#8617; Go back and read the code cell</a>
</nav>

## Interview Question 1

**The Hiring Manager asks:** *'I see you used PCA to visualize the data. Print the variance explained ratio — did we lose too much information? If we retained only 65% of the variance, is this 2D scatter plot still trustworthy?'*

**Your Answer:**

- **Variance Retained:** *[State your actual variance retained percentage]*

- **Trustworthiness:** *[Discuss what losing 35% of variance means in practice — is the overall cluster structure preserved even if fine-grained distances are distorted?]*

- **Trade-off:** *[Is the 2D visualization useful as an exploratory tool even if imperfect? What would you do before finalizing cluster assignments — work on the full 5D scaled data or the 2D projection?]*

---
**YOUR ANSWER HERE**

---
## Task 2: K-Means Clustering

**Context:** We need to group the fans. We do not know K in advance.

<a name="read-code-11"></a>

**Task 2 Code Cell — Find Optimal K Using Elbow Curve and Silhouette Score**

Step 1: Run K-Means for k = 2 through k = 10 on `X_scaled`. Record the inertia (WCSS — within-cluster sum of squares) for each k. Step 2: Plot the Elbow Curve (inertia vs k). Look for the point where adding another cluster yields diminishing returns. Step 3: Calculate the Silhouette Score for k = 3, 4, and 5. Then fit your final K-Means model and add the cluster labels to `df_fans`.

<nav aria-label="Code cell 11 navigation">
<a href="#skip-code-11" aria-label="Skip code cell 11 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# TODO: Elbow Curve
inertias = []
k_range = range(2, 11)

# for k in k_range:
#     km = KMeans(n_clusters=k, random_state=42, n_init=10)
#     km.fit(X_scaled)
#     inertias.append(km.inertia_)
# [YOUR CODE HERE — fill in the loop]

# plt.figure(figsize=(10, 5))
# plt.plot(k_range, inertias, marker='o', linewidth=2, color='steelblue')
# plt.xlabel('Number of Clusters (K)')
# plt.ylabel('Inertia (WCSS)')
# plt.title('Elbow Curve — Fan Segmentation')
# plt.xticks(k_range)
# plt.grid(alpha=0.3)
# plt.show()

# TODO: Silhouette Scores for k=3, 4, 5
# for k in [3, 4, 5]:
#     km = KMeans(n_clusters=k, random_state=42, n_init=10)
#     labels = km.fit_predict(X_scaled)
#     score = silhouette_score(X_scaled, labels)
#     print(f'k={k}: Silhouette Score = {score:.4f}')

# TODO: Apply final K-Means (choose K based on elbow + silhouette)
# best_k = ???  # YOUR CHOICE
# km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
# df_fans['Cluster'] = km_final.fit_predict(X_scaled)
# print(f'\nFinal K={best_k}. Cluster value counts:')
# print(df_fans['Cluster'].value_counts())

<a name="skip-code-11"></a>

**Expected output:** An elbow curve showing a clear bend, three silhouette scores (higher is better — maximum is 1.0), and a cluster value count. The elbow and silhouette evidence should both point toward K = 4, reflecting the four distinct groups generated in the data.

<nav aria-label="Return navigation for code cell 11">
<a href="#read-code-11" aria-label="Go back and read code cell 11">&#8617; Go back and read the code cell</a>
</nav>

---
## Task 3: Cluster Interpretation — The Business Part

**Context:** Reporting 'Cluster 0, 1, 2, 3' means nothing to the marketing director. Your job is to make the data speak.

**Interview Tip:** In the real world, this is often the hardest and most valuable part. Any data scientist can run K-Means. Fewer can translate cluster mean profiles into actionable marketing personas.

<a name="read-code-12"></a>

**Task 3 Code Cell — Profile Each Cluster**

This cell groups `df_fans` by `Cluster` and computes the mean of every feature. Examine the differences carefully. Which cluster has the highest `Merch_Spend_USD`? The highest `Content_Requests`? The highest `Support_Contacts`? Use these profiles to answer Interview Question 2.

<nav aria-label="Code cell 12 navigation">
<a href="#skip-code-12" aria-label="Skip code cell 12 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# TODO: Profile clusters by computing group means
# cluster_profiles = df_fans.groupby('Cluster').mean().round(2)
# print('Cluster Mean Profiles:')
# print(cluster_profiles)

# Optional: Heatmap for visual comparison
# plt.figure(figsize=(10, 5))
# sns.heatmap(cluster_profiles.T, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Mean Value'})
# plt.title('Fan Cluster Profile Heatmap')
# plt.xlabel('Cluster')
# plt.ylabel('Feature')
# plt.tight_layout()
# plt.savefig('fig_cluster_profiles.png', dpi=150, bbox_inches='tight')
# plt.show()

<a name="skip-code-12"></a>

**Expected output:** A table of mean feature values per cluster and optionally a heatmap. Look for one cluster with very high `Content_Requests` and near-zero everything else — those are the bots. Look for one with high `Merch_Spend_USD` and high `Events_Attended` — those are the Super Fans.

**Accessibility note:** Add a Markdown cell below describing the heatmap: which cluster stands out most in each feature, and which cluster you believe corresponds to bots.

<nav aria-label="Return navigation for code cell 12">
<a href="#read-code-12" aria-label="Go back and read code cell 12">&#8617; Go back and read the code cell</a>
</nav>

## Interview Question 2

**The Hiring Manager asks:** *'Based on the cluster profiles, give each cluster a marketing persona name and explain your reasoning.'*

**Your Answer:**

- **Cluster 0:** Name = *[Name]* | Reason: *[Which feature values define this group?]*

- **Cluster 1:** Name = *[Name]* | Reason: *[e.g., High Merch_Spend_USD and Events_Attended = Super Fan]*

- **Cluster 2:** Name = *[Name]* | Reason: *...*

- **Cluster 3:** Name = *[Name]* | Reason: *...*

---
**YOUR ANSWER HERE**

---
## Task 4: Anomaly Detection — DBSCAN

**Context:** K-Means forces every point into a cluster. Bots are behavioral outliers — they operate in a tight, dense region far outside normal fan behavior (near-zero session time, hundreds of content requests, zero purchases). DBSCAN can identify them as **noise (label = −1)** rather than forcing them into a fan segment.

<a name="read-code-13"></a>

**Task 4 Code Cell — Apply DBSCAN for Bot Detection**

Step 1: Fit `DBSCAN(eps=0.5, min_samples=5)` to `X_scaled`. Step 2: Plot PC1 vs PC2 colored by DBSCAN label. Points labeled −1 are 'noise' — in this dataset, these are the likely bot accounts. Step 3: Print how many points received each label, including how many are noise (−1).

<nav aria-label="Code cell 13 navigation">
<a href="#skip-code-13" aria-label="Skip code cell 13 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# TODO: DBSCAN — Bot Detection
# db = DBSCAN(eps=0.5, min_samples=5)
# db_labels = db.fit_predict(X_scaled)
# [YOUR CODE HERE]

# Print label distribution
# unique_labels, counts = np.unique(db_labels, return_counts=True)
# print('DBSCAN Label Distribution:')
# for label, count in zip(unique_labels, counts):
#     name = 'Noise (Bots)' if label == -1 else f'Cluster {label}'
#     print(f'  {name}: {count} accounts')

# Plot DBSCAN result on 2D PCA
# plt.figure(figsize=(10, 7))
# palette = {-1: 'red', 0: 'steelblue', 1: 'green', 2: 'orange', 3: 'purple'}
# for label in np.unique(db_labels):
#     mask = db_labels == label
#     name = 'NOISE (Bots)' if label == -1 else f'Cluster {label}'
#     plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=name,
#                 alpha=0.6, s=20, c=palette.get(label, 'gray'))
# plt.xlabel('PC1'); plt.ylabel('PC2')
# plt.title('DBSCAN Fan Segmentation — Noise = Potential Bots')
# plt.legend()
# plt.grid(alpha=0.3)
# plt.tight_layout()
# plt.savefig('fig_dbscan.png', dpi=150, bbox_inches='tight')
# plt.show()

<a name="skip-code-13"></a>

**Expected output:** DBSCAN label counts showing approximately 50 noise points (the bots), and a scatter plot where the noise points (red) form a tight cluster isolated from the main fan behavioral groups.

**If DBSCAN merges everything or produces too many noise points:** Try adjusting `eps` (larger = more generous neighborhood) or `min_samples` (smaller = looser density requirement). This tuning is part of the answer to Interview Question 3.

**Accessibility note:** Add a Markdown cell below describing the scatter plot. Note where the red noise points appear relative to the other clusters.

<nav aria-label="Return navigation for code cell 13">
<a href="#read-code-13" aria-label="Go back and read code cell 13">&#8617; Go back and read the code cell</a>
</nav>

## Interview Question 3

**The Hiring Manager asks:**

*'In K-Means, every fan must belong to a cluster. In DBSCAN, some are labeled −1.*
*1. What does −1 represent in this specific fan intelligence context?*
*2. If you were building a real-time bot detection system for live streaming events, would you use K-Means or DBSCAN? Why?'*

**Your Answer:**

- **What −1 means here:** *[Points that do not have enough neighbors within distance eps to form a dense region — in this context, accounts behaving so differently from normal fans that no cluster claims them]*

- **K-Means vs DBSCAN for bot detection:** *[K-Means assigns every point to a centroid — bots get forced into the nearest fan cluster, masking them. DBSCAN isolates behavioral outliers as noise, making them easy to flag. For real-time fraud/bot detection, density-based methods are generally preferred.]*

- **DBSCAN limitation for production:** *[DBSCAN has no built-in `predict()` method for new data — discuss how you would handle this in a live streaming context.]*

---
**YOUR ANSWER HERE**

---
# Part 3: Reinforcement Learning

## Scenario

**SolarVault** has deployed residential battery storage systems in 10,000 homes. Each home has rooftop solar panels and a battery. The goal is to:

- **Charge the battery** when solar production is high and grid prices are low
- **Sell stored energy back to the grid** when prices peak (typically early evening)
- **Hold** when neither action is profitable

**The problem is that we do not have a labeled dataset of 'correct actions.'** The optimal strategy depends on the time of day, current battery level, and live market prices. We need an agent that *learns* the strategy from experience.

**Message from the CTO:**

> 'Build a Q-Learning agent that learns energy arbitrage. Explain how we would scale this to 10,000 homes with different solar profiles without violating homeowner privacy.'

---

<a name="read-code-14"></a>

**Setup Cell — Import Libraries (Part 3)**

This cell imports numpy, pandas, matplotlib, and seaborn. No scikit-learn is needed for Part 3 — we implement Q-Learning from scratch.

<nav aria-label="Code cell 14 navigation">
<a href="#skip-code-14" aria-label="Skip code cell 14 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_style('whitegrid')
%matplotlib inline
print('Part 3 setup complete.')

<a name="skip-code-14"></a>

**Expected output:** `Part 3 setup complete.`

<nav aria-label="Return navigation for code cell 14">
<a href="#read-code-14" aria-label="Go back and read code cell 14">&#8617; Go back and read the code cell</a>
</nav>

---
## Task 1: The Environment

We model the battery as an **OpenAI Gym-style environment**.

- **State:** `(current_charge_level, hour_of_day)` — charge is 0 to 10 units, hour is 0 to 23
- **Actions:** 0 = Hold, 1 = Charge (buy from grid), 2 = Discharge (sell to grid)
- **Reward:** Revenue from selling minus cost of buying; penalties for invalid actions

The 24-hour price cycle reflects real grid dynamics: low prices overnight, a morning ramp, a midday plateau, and a sharp evening peak when solar production drops but demand surges.

<a name="read-code-15"></a>

**Task 1 Code Cell — Market Data and BatteryEnv Class (Do Not Modify)**

This cell defines the 24-hour electricity price cycle and the `BatteryEnv` environment class. The class has three methods: `reset()` (start a new episode), `step(action)` (execute an action and receive the next state, reward, and done flag). Read the class carefully before implementing the Q-Learning loop in Task 2.

<nav aria-label="Code cell 15 navigation">
<a href="#skip-code-15" aria-label="Skip code cell 15 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# 24-hour electricity price cycle ($/MWh)
# Low overnight, morning ramp, midday plateau, sharp evening peak
prices = np.array([8, 8, 8, 9, 12, 18, 28, 42, 55, 48, 38, 32,
                   28, 22, 18, 16, 22, 38, 75, 95, 88, 55, 28, 12])

plt.figure(figsize=(10, 4))
plt.plot(prices, marker='o', linestyle='--', color='darkorange', linewidth=2)
plt.fill_between(range(24), prices, alpha=0.15, color='darkorange')
plt.title('24-Hour Grid Electricity Price Cycle ($/MWh)')
plt.xlabel('Hour of Day'); plt.ylabel('Price ($/MWh)')
plt.xticks(range(24))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_prices.png', dpi=150, bbox_inches='tight')
plt.show()

class BatteryEnv:
    def __init__(self, prices):
        self.prices       = prices
        self.max_capacity = 10
        self.current_charge = 0
        self.time_step    = 0
        self.end_step     = len(prices) - 1

    def reset(self):
        self.current_charge = 0
        self.time_step      = 0
        return (self.current_charge, self.time_step)

    def step(self, action):
        current_price = self.prices[self.time_step]
        reward = 0

        if action == 1:   # CHARGE — buy from grid
            if self.current_charge < self.max_capacity:
                self.current_charge += 1
                reward = -current_price
            else:
                reward = -50  # Penalty: already full

        elif action == 2: # DISCHARGE — sell to grid
            if self.current_charge > 0:
                self.current_charge -= 1
                reward = current_price
            else:
                reward = -50  # Penalty: battery empty
        # action == 0: Hold, reward stays 0

        self.time_step += 1
        done = (self.time_step >= self.end_step)
        next_state = (self.current_charge, self.time_step)
        return next_state, reward, done

print('BatteryEnv class defined.')

<a name="skip-code-15"></a>

**Expected output:** A price curve plot and `BatteryEnv class defined.`

The price curve should show the classic 'duck curve' shape: low overnight prices (ideal for charging), a moderate midday period, and a sharp peak around hour 18–20 (ideal for discharging/selling).

**Accessibility note:** Add a Markdown cell below describing the price curve. Note the hours of lowest price, the steepness of the evening ramp, and the peak hour.

<nav aria-label="Return navigation for code cell 15">
<a href="#read-code-15" aria-label="Go back and read code cell 15">&#8617; Go back and read the code cell</a>
</nav>

---
## Task 2: Build the Agent — Q-Learning

**Context:** Q-Learning is a model-free reinforcement learning algorithm. The agent maintains a Q-table: a lookup of expected future reward for every (state, action) pair. It updates this table using the **Bellman Equation**:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ R + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Where:
- $\alpha$ = learning rate (how fast to update)
- $\gamma$ = discount factor (how much to value future vs immediate reward)
- $R$ = immediate reward received
- $\max Q(s', a')$ = best possible future reward from the next state

<a name="read-code-16"></a>

**Task 2 Code Cell — Q-Learning Training Loop**

The hyperparameters, Q-table initialization, and episode loop structure are provided. Your task is to fill in the **Q-table update line** (the Bellman update). After training, the cell plots the total reward per episode — a rising curve that eventually plateaus indicates the agent is learning.

<nav aria-label="Code cell 16 navigation">
<a href="#skip-code-16" aria-label="Skip code cell 16 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Hyperparameters
alpha   = 0.1    # Learning rate
gamma   = 0.95   # Discount factor — agent cares strongly about future profit
epsilon = 0.1    # Exploration rate — 10% of actions are random
episodes = 1000

# Q-table: [Charge_Level(0-10), Hour(0-23), Actions(3)]
q_table = np.zeros((11, 24, 3))

env = BatteryEnv(prices)
rewards_history = []

for ep in range(episodes):
    state      = env.reset()
    done       = False
    total_reward = 0

    while not done:
        charge, hour = state

        # Epsilon-Greedy Action Selection
        if np.random.uniform(0, 1) < epsilon:
            action = np.random.choice([0, 1, 2])  # Explore
        else:
            action = np.argmax(q_table[charge, hour])  # Exploit

        # Take action
        next_state, reward, done = env.step(action)
        next_charge, next_hour   = next_state

        # TODO: Bellman Update — fill in this line
        # old_value   = q_table[charge, hour, action]
        # next_max    = np.max(q_table[next_charge, next_hour]) if not done else 0
        # q_table[charge, hour, action] = ???

        state        = next_state
        total_reward += reward

    rewards_history.append(total_reward)

# Plot learning curve
plt.figure(figsize=(12, 5))
plt.plot(rewards_history, alpha=0.4, color='steelblue', label='Episode Reward')
plt.plot(pd.Series(rewards_history).rolling(50).mean(), color='darkorange',
         linewidth=2, label='50-Episode Rolling Mean')
plt.title('SolarVault Agent — Profit per Episode During Training')
plt.xlabel('Episode'); plt.ylabel('Total Profit ($)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('fig_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Final 100-episode average profit: ${np.mean(rewards_history[-100:]):.2f}')

<a name="skip-code-16"></a>

**Expected output after filling in the Bellman update:**

The Bellman update line should be:
`q_table[charge, hour, action] = old_value + alpha * (reward + gamma * next_max - old_value)`

The learning curve plot should show the 50-episode rolling mean rising from a low/negative value and stabilizing around a positive profit plateau after approximately 200–400 episodes. The raw per-episode line will remain noisy due to epsilon-greedy exploration.

**Accessibility note:** Add a Markdown cell describing the learning curve — where does the rolling mean stabilize? What does this tell you about convergence?

<nav aria-label="Return navigation for code cell 16">
<a href="#read-code-16" aria-label="Go back and read code cell 16">&#8617; Go back and read the code cell</a>
</nav>

## Interview Question 1

**The CTO asks:** *'The training curve rises and then flattens but never becomes perfectly smooth. Why?*
*And if we deployed this to a real grid where prices spike unpredictably (unlike this fixed 24-hour cycle), what would break and how would you fix it?'*

**Your Answer:**

- **Why the curve stays noisy:** *[The epsilon = 0.1 exploration rate means 10% of actions are random in every episode, injecting permanent variance into rewards. Discuss epsilon decay as a solution.]*

- **What breaks with real price spikes:** *[The current state is (charge_level, hour) — it does not include the current price. An unexpected price spike at hour 14 looks identical to a normal hour 14. Discuss adding `current_price` or a short price history window to the state.]*

- **How to handle continuous states:** *[The Q-table approach requires discretized states. Discuss Deep Q-Networks (DQN) as a scalable alternative for continuous price inputs.]*

---
**YOUR ANSWER HERE**

---
## Task 3: Scaling to 10,000 Homes — Transfer and Federated Learning

**Context:** SolarVault wants to deploy this system to 10,000 residential homes.

- **Problem A:** Training a model from scratch for every house takes too long (each house has different solar panel orientation, local weather, and usage patterns).
- **Problem B:** Homeowners are uncomfortable sharing their energy usage data — it reveals when they are home, on vacation, and their daily routines.

---

## Interview Question 2 — Transfer Learning

**The CTO asks:** *'We have a fully trained agent for House A in Phoenix (Southwest, high sun hours). House B in Seattle (Northwest, frequent cloud cover) comes online tomorrow. Instead of starting from random initialization for House B, how would you use Transfer Learning to speed up training? If we converted this to a Deep Q-Network, which layers would you freeze and which would you retrain?'*

**Your Answer:**

- **Concept — Warm Start:** *[Initialize House B's Q-table (or network weights) using House A's trained values rather than zeros. General grid dynamics — charge low, sell high — transfer across homes.]*

- **What transfers:** *[The general strategy: buy at low prices, sell at peak. This logic holds in both Phoenix and Seattle.]*

- **What needs retraining:** *[Seattle's solar production is lower and more variable. The state-value estimates for 'midday = high charge opportunity' will be too optimistic. If using a DQN: freeze early layers (general grid dynamics), retrain the final layers (local solar and weather patterns).]*

---
**YOUR ANSWER HERE**

---

## Interview Question 3 — Federated Learning

**The CTO asks:** *'Explain to the Legal and Privacy team how Federated Learning solves the homeowner data concern. Specifically: what leaves each home, what stays local, and what happens in the cloud?'*

**Your Answer:**

- **What stays on the device:** *[Raw energy usage data — when the homeowner is home, charging patterns, consumption history — never leaves the household device.]*

- **What is sent to the cloud:** *[Only the model weight updates (gradients) computed locally from the homeowner's own data. These updates contain statistical patterns but not raw observations.]*

- **What happens in the cloud:** *[FedAvg (Federated Averaging) aggregates weight updates from all 10,000 homes into a single improved global model. The improved model is then pushed back to each device.]*

- **Remaining privacy risk:** *[Gradient inversion attacks can sometimes reconstruct approximate training data from gradients. Discuss differential privacy (adding calibrated noise to gradients) as a mitigation.]*

---
**YOUR ANSWER HERE**

---
# Congratulations!

You have completed the Machine Learning Paradigms simulation.

## What You Practiced

**Supervised Learning (Part 1 — SportsPulse Analytics)**
- Detecting OLS assumption violations from residual plots
- Diagnosing accuracy as a misleading metric under class imbalance
- Computing asymmetric business cost at different decision thresholds
- Interpreting Random Forest feature importances and their directional limitations

**Unsupervised Learning (Part 2 — SportsPulse Fan Intelligence)**
- Why scaling is mandatory before distance-based algorithms
- Selecting K using the Elbow method and Silhouette Score
- Translating numeric cluster profiles into actionable business personas
- Using DBSCAN to isolate anomalies that K-Means forces into clusters

**Reinforcement Learning (Part 3 — SolarVault)**
- Implementing the Bellman update equation
- Diagnosing the exploration-exploitation tradeoff
- Explaining Transfer Learning to accelerate cross-domain deployment
- Explaining Federated Learning to a privacy-conscious audience

## Interview Preparation Checklist

- Can you name two OLS assumptions and explain what a non-random residual pattern implies?
- Can you explain why a 90% accuracy on an imbalanced dataset is meaningless?
- Can you calculate business cost under different thresholds from a confusion matrix?
- Can you explain what Random Forest feature importance measures — and what it cannot tell you?
- Can you explain why K-Means fails before scaling?
- Can you explain the difference between inertia and silhouette score?
- Can you name two situations where DBSCAN is preferable to K-Means?
- Can you write the Bellman update equation from memory?
- Can you explain Transfer Learning and Federated Learning to a non-technical executive?

## The Core Interview Principle

> The interviewer cares more about your reasoning than your accuracy score. Explain your assumptions, your choices, and especially your trade-offs.

---

**To share:** Go to **Share** in Colab → set access to **Anyone with the link can view** → copy the link → paste into Canvas.